In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from datasets import load_dataset
import os

import numpy as np
import pandas as pd
import warnings
import os

# 1. ОТКЛЮЧАЕМ ПРЕДУПРЕЖДЕНИЯ И СНИМАЕМ ЛИМИТЫ
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
pd.set_option('display.max_columns', None) # Видеть все колонки
pd.set_option('display.expand_frame_repr', False)

In [3]:
import os
import pandas as pd
import numpy as np
import torch
import warnings
from datasets import load_dataset

# 1. Настройки вывода и игнорирование ворнингов
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# 2. Проверка GPU (в Колабе это обязательно)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")
if torch.cuda.is_available():
    print(f"Модель GPU: {torch.cuda.get_device_name(0)}")

# 3. Загрузка данных
print("Загрузка датасета из Hugging Face...")
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df_full = pd.DataFrame(dataset['train'])

data_path = "/content/drive/MyDrive/"

def load_indices(filename):
    full_path = os.path.join(data_path, filename)
    if not os.path.exists(full_path):
        # Если файлов нет, создадим заглушку для теста или выведем ошибку
        raise FileNotFoundError(f"Файл {filename} не найден в {data_path}. Загрузи его в панель слева!")
    with open(full_path, 'r') as f:
        return [int(line.strip()) for line in f]

try:
    print("Загрузка индексов разбиения...")
    train_idx = load_indices("train_idx.txt")
    val_idx = load_indices("val_idx.txt")
    test_idx = load_indices("test_idx.txt")

    train = df_full.iloc[train_idx].copy()
    val = df_full.iloc[val_idx].copy()
    test = df_full.iloc[test_idx].copy()

    print(f"Успешно! Размеры: Train={len(train)}, Val={len(val)}, Test={len(test)}")
except Exception as e:
    print(f"Ошибка: {e}")

# 4. Объединяем текст (как мы решили — без лемматизации, с разделителем)
def preprocess_for_bert(df):
    return (df['subject'].fillna("") + " [SEP] " + df['body'].fillna("")).astype(str)

for df in [train, val, test]:
    df['combined_text'] = preprocess_for_bert(df)

# 5. Кодируем таргеты (BERT нужны числа)
from sklearn.preprocessing import LabelEncoder
target_cols = ['queue', 'priority', 'type']
encoders = {col: LabelEncoder() for col in target_cols}

for col in target_cols:
    train[f'{col}_label'] = encoders[col].fit_transform(train[col].astype(str))
    val[f'{col}_label'] = encoders[col].transform(val[col].astype(str))
    test[f'{col}_label'] = encoders[col].transform(test[col].astype(str))

print("\nПодготовка завершена. Мы готовы к созданию Dataset и DataLoader.")

Используем устройство: cuda
Модель GPU: Tesla T4
Загрузка датасета из Hugging Face...


README.md: 0.00B [00:00, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…):   0%|          | 0.00/26.0M [00:00<?, ?B/s]

(…)set-tickets-german_normalized_50_5_2.csv: 0.00B [00:00, ?B/s]

dataset-tickets-multi-lang-4-20k.csv:   0%|          | 0.00/18.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

Загрузка индексов разбиения...
Успешно! Размеры: Train=49412, Val=6176, Test=6177

Подготовка завершена. Мы готовы к созданию Dataset и DataLoader.


In [4]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class TicketDataset(Dataset):
    def __init__(self, texts, labels_queue, labels_priority, labels_type, tokenizer, max_len):
        self.texts = texts.values
        self.labels_queue = labels_queue.values
        self.labels_priority = labels_priority.values
        self.labels_type = labels_type.values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])

        # Вместо encode_plus используем прямой вызов токенизатора
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt' # Сразу просим тензоры PyTorch
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'queue_label': torch.tensor(self.labels_queue[item], dtype=torch.long),
            'priority_label': torch.tensor(self.labels_priority[item], dtype=torch.long),
            'type_label': torch.tensor(self.labels_type[item], dtype=torch.long)
        }

# --- ИНИЦИАЛИЗАЦИЯ ---

# 1. Выбираем модель (начнем с легкой)
# MODEL_NAME = 'distilbert-base-multilingual-cased'
MODEL_NAME = 'microsoft/Multilingual-MiniLM-L12-H384'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Параметры обучения
MAX_LEN = 128  # Для начала хватит, чтобы не упасть по памяти
BATCH_SIZE = 8

train_dataset = TicketDataset(
    texts=train['combined_text'],
    labels_queue=train['queue_label'],
    labels_priority=train['priority_label'],
    labels_type=train['type_label'],
    tokenizer=tokenizer, max_len=MAX_LEN
)

val_dataset = TicketDataset(
    texts=val['combined_text'],
    labels_queue=val['queue_label'],
    labels_priority=val['priority_label'],
    labels_type=val['type_label'],
    tokenizer=tokenizer, max_len=MAX_LEN
)

test_dataset = TicketDataset(
    texts=test['combined_text'],
    labels_queue=test['queue_label'],
    labels_priority=test['priority_label'],
    labels_type=test['type_label'],
    tokenizer=tokenizer, max_len=MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"DataLoaders готовы: Train({len(train_loader)}), Val({len(val_loader)}), Test({len(test_loader)})")
print(f"Dataset создан. В одном батче: {BATCH_SIZE} примеров.")

config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

DataLoaders готовы: Train(6177), Val(772), Test(773)
Dataset создан. В одном батче: 8 примеров.


In [5]:
# Извлекаем первый батч из загрузчика
sample_batch = next(iter(train_loader))

print(f"Формат input_ids: {sample_batch['input_ids'].shape}") # Должно быть [16, 128]
print(f"Формат маски: {sample_batch['attention_mask'].shape}") # Должно быть [16, 128]
print(f"Пример таргетов (queue): {sample_batch['queue_label'][:5]}") # Первые 5 ответов

# Декодируем самый первый текст из батча, чтобы увидеть его "глазами модели"
decoded_text = tokenizer.decode(sample_batch['input_ids'][0])
print("\nДекодированный текст первого письма:")
print(decoded_text[:200] + "...")

Формат input_ids: torch.Size([8, 128])
Формат маски: torch.Size([8, 128])
Пример таргетов (queue): tensor([17, 39, 10, 39, 30])

Декодированный текст первого письма:
<s> Meldung über Ausfallzeiten des Dexcom G6-Systems [SEP] Sehr geehrtes Support-Team,\n\nwir haben derzeit einen schweren Systemausfall, der Dexcom G6 betrifft, was für die Echtzeit-Glukoseüberwachun...


In [6]:
import torch.nn as nn
import torch.optim as optim # Добавляем стандартный оптимизатор PyTorch
from transformers import AutoModel, get_linear_schedule_with_warmup
import torch
import copy

# АРХИТЕКТУРА
class MultiTaskTicketClassifier(nn.Module):
    def __init__(self, model_name, num_classes_dict):
        super(MultiTaskTicketClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Три головы для трех задач
        self.queue_head = nn.Linear(hidden_size, num_classes_dict['queue'])
        self.priority_head = nn.Linear(hidden_size, num_classes_dict['priority'])
        self.type_head = nn.Linear(hidden_size, num_classes_dict['type'])
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[0][:, 0, :] # Берем CLS токен
        pooled_output = self.dropout(pooled_output)

        return self.queue_head(pooled_output), self.priority_head(pooled_output), self.type_head(pooled_output)

# ИНИЦИАЛИЗАЦИЯ ПАРАМЕТРОВ
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EARLY_STOP_PATIENCE = 3
best_score = 0
patience_counter = 0

# Словарь количества классов (берем из твоих энкодеров)
num_classes_dict = {
    'queue': len(encoders['queue'].classes_),
    'priority': len(encoders['priority'].classes_),
    'type': len(encoders['type'].classes_)
}

# СОЗДАНИЕ ОБЪЕКТА МОДЕЛИ (Исправляет твой NameError)
model = MultiTaskTicketClassifier(MODEL_NAME, num_classes_dict)
model.to(device)

# СОХРАНЯЕМ ВЕСА СРАЗУ (чтобы переменная существовала)
best_model_wts = copy.deepcopy(model.state_dict()) # <--- ДОБАВИТЬ СЮДА

# Оптимизатор и Лосс
# Используем оптимизатор из torch.optim вместо transformers
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss().to(device)

# Планировщик
EPOCHS = 20
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

print(f"Модель создана и готова! Устройство: {device}")

pytorch_model.bin:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Модель создана и готова! Устройство: cuda


In [7]:
from tqdm.auto import tqdm # Прогресс-бар
from sklearn.metrics import f1_score, accuracy_score

def train_epoch(model, data_loader, criterion, optimizer, device, scheduler):
    model.train()
    losses = []

    # Добавляем tqdm для визуализации обучения
    progress_bar = tqdm(data_loader, desc='Training', leave=False)

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        q_true = batch['queue_label'].to(device)
        p_true = batch['priority_label'].to(device)
        t_true = batch['type_label'].to(device)

        optimizer.zero_grad()
        q_logits, p_logits, t_logits = model(input_ids, attention_mask)

        # Считаем лоссы (применяем твои веса 0.7 / 0.15 / 0.15 к градиентам)
        loss_q = criterion(q_logits, q_true)
        loss_p = criterion(p_logits, p_true)
        loss_t = criterion(t_logits, t_true)

        total_loss = 0.70 * loss_q + 0.15 * loss_p + 0.15 * loss_t
        losses.append(total_loss.item())

        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        # Обновляем инфо в прогресс-баре
        progress_bar.set_postfix({'loss': f'{total_loss.item():.4f}'})

    return np.mean(losses)

def evaluate(model, data_loader, criterion, device, desc="Evaluating"):
    model.eval()
    losses = []
    q_all, p_all, t_all = [], [], []
    q_preds, p_preds, t_preds = [], [], []

    progress_bar = tqdm(data_loader, desc=desc, leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            q_true = batch['queue_label'].to(device)
            p_true = batch['priority_label'].to(device)
            t_true = batch['type_label'].to(device)

            q_logits, p_logits, t_logits = model(input_ids, attention_mask)

            # Считаем лосс для статистики
            loss_q = criterion(q_logits, q_true)
            loss_p = criterion(p_logits, p_true)
            loss_t = criterion(t_logits, t_true)
            total_loss = 0.70 * loss_q + 0.15 * loss_p + 0.15 * loss_t
            losses.append(total_loss.item())

            # Собираем предсказания
            q_preds.extend(torch.argmax(q_logits, dim=1).cpu().numpy())
            p_preds.extend(torch.argmax(p_logits, dim=1).cpu().numpy())
            t_preds.extend(torch.argmax(t_logits, dim=1).cpu().numpy())

            q_all.extend(batch['queue_label'].numpy())
            p_all.extend(batch['priority_label'].numpy())
            t_all.extend(batch['type_label'].numpy())

    m_f1_q = f1_score(q_all, q_preds, average='macro')
    acc_p = accuracy_score(p_all, p_preds)
    acc_t = accuracy_score(t_all, t_preds)
    total_score = 0.70 * m_f1_q + 0.15 * acc_p + 0.15 * acc_t

    return {
        'loss': np.mean(losses),
        'score': total_score,
        'q_f1': m_f1_q,
        'p_acc': acc_p,
        't_acc': acc_t
    }

# --- 3. ОБНОВЛЕННЫЙ ЦИКЛ ОБУЧЕНИЯ ---

for epoch in range(EPOCHS):
    print(f"\n{'='*30} Эпоха {epoch + 1}/{EPOCHS} {'='*30}")

    # 1. Обучение
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scheduler)

    # 2. Оценка на всех датасетах
    train_metrics = evaluate(model, train_loader, criterion, device, desc="Val on Train")
    val_metrics = evaluate(model, val_loader, criterion, device, desc="Val on Valid")
    test_metrics = evaluate(model, test_loader, criterion, device, desc="Val on Test")

    # 3. ПЕЧАТЬ ТАБЛИЦЫ РЕЗУЛЬТАТОВ
    print(f"\nОтчет по эпохе {epoch + 1}:")
    data = [
        ["Train", train_metrics['loss'], train_metrics['q_f1'], train_metrics['p_acc'], train_metrics['t_acc'], train_metrics['score']],
        ["Valid", val_metrics['loss'], val_metrics['q_f1'], val_metrics['p_acc'], val_metrics['t_acc'], val_metrics['score']],
        ["Test", test_metrics['loss'], test_metrics['q_f1'], test_metrics['p_acc'], test_metrics['t_acc'], test_metrics['score']]
    ]

    print(f"{'Dataset':<10} | {'Loss':<7} | {'Q_F1':<7} | {'P_Acc':<7} | {'T_Acc':<7} | {'SCORE':<7}")
    print("-" * 65)
    for row in data:
        print(f"{row[0]:<10} | {row[1]:.4f}  | {row[2]:.4f}  | {row[3]:.4f}  | {row[4]:.4f}  | {row[5]:.4f}")

    # 4. Early Stopping (следим по VALIDATION score)
    if val_metrics['score'] > best_score:
        best_score = val_metrics['score']
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"\n>>> Улучшение! Best Valid Score: {best_score:.4f}. Модель сохранена.")
    else:
        patience_counter += 1
        print(f"\n>>> Нет улучшений на Valid. Терпение: {patience_counter}/{EARLY_STOP_PATIENCE}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print("\n!!! EARLY STOPPING TRIGGERED !!!")
        break

# В самом конце восстанавливаем лучшую модель
model.load_state_dict(best_model_wts)
print("\nОбучение завершено. Лучшие веса загружены.")


============================== Эпоха 1/20 ==============================


Training:   0%|          | 0/6177 [00:00<?, ?it/s]

Val on Train:   0%|          | 0/6177 [00:00<?, ?it/s]

Val on Valid:   0%|          | 0/772 [00:00<?, ?it/s]

Val on Test:   0%|          | 0/773 [00:00<?, ?it/s]

...

Val on Valid:   0%|          | 0/772 [00:00<?, ?it/s]

Val on Test:   0%|          | 0/773 [00:00<?, ?it/s]


Отчет по эпохе 12:
Dataset    | Loss    | Q_F1    | P_Acc   | T_Acc   | SCORE  
-----------------------------------------------------------------
Train      | 0.3335  | 0.9735  | 0.6178  | 0.8660  | 0.9041
Valid      | 1.0995  | 0.8733  | 0.5656  | 0.8489  | 0.8235
Test       | 1.1035  | 0.8691  | 0.5584  | 0.8477  | 0.8193

>>> Улучшение! Best Valid Score: 0.8235. Модель сохранена.

============================== Эпоха 13/20 ==============================


Training:   0%|          | 0/6177 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# учится норм и стабильно достаточно, проблема в том что ресурсы колаба ограничены

In [ ]:
# берт уперся в свой потолок 0.82